# SONARA Run 2, first pass: pretrained weights, input normalization, training recipe

Run 1 (`results/run1_baseline_2026-10-12`) is the baseline. This notebook tests three changes that may improve on it, one step at a time, **using validation data only**.

| Change | What it does |
| :- | :- |
| **Input normalization** | Each clip's dB features are rescaled to zero mean and unit spread, per channel. Run 1 fed raw dB values. |
| **Pretrained weights** | MobileNetV3-Small and EfficientNet-B0 start from ImageNet weights instead of random ones. |
| **Training recipe** | AdamW, a higher learning rate with warm-up and a cosine schedule, and mixup (blending pairs of training clips). Run 1 used Adam at 1e-4, untuned. |

## Rules that keep the result honest

1. **A test set is locked away first.** 15% of SmartEars, grouped by source recording, is set aside in Part 3 and saved to `locked_test_set.csv`. **No cell in this notebook trains on it, evaluates on it, or reads its labels for any decision.** PVSD is not loaded at all.
2. **Everything here is a validation result.** The remaining 85% is split into training and validation data, again grouped by recording, with a new split for each seed.
3. **A change counts only if it helps on most seeds**, compared with the setup it was added to.
4. **The selection rule is fixed in advance:** the highest mean validation ROC-AUC among models that can run on the Raspberry Pi 4.
5. **The locked test set and PVSD are opened once**, in a later notebook, for the final chosen setup and the Run 1 CNN–BiLSTM.

Validation numbers are slightly optimistic, because early stopping and the threshold also use the validation set. That affects every setup equally, so the comparison is fair, but these are **not** numbers for the paper's results table.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. The notebook asks for Google Drive access so results survive a disconnect. Expect roughly 4 to 5 hours for 9 setups × 3 seeds; finished runs are skipped on rerun.

## 1. Setup and configuration

In [1]:
import os, re, copy, glob, hashlib, json, math, random, time
from math import gcd

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, LambdaLR

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__, "| torchvision", torchvision.__version__)

Device: cuda | torch 2.11.0+cu130 | torchvision 0.26.0+cu130


In [2]:
# ---- Data and outputs ----------------------------------------------------
REPO_URL = "https://github.com/tagalogclark123-cmyk/SONARA-GITHUB.git"
REPO_DIR = "/content/SONARA-GITHUB"
SAVE_TO_DRIVE = True          # keep results in Google Drive so a disconnect loses nothing
OUT_DIR = "/content/drive/MyDrive/SONARA/run2_first_pass" if SAVE_TO_DRIVE else "/content/run2_first_pass"
SMARTEARS_ROOT = "Smartears"
LABEL_MAP = {"Healthy": "Normal", "None": "Noise", "Sick": "Distress"}

# ---- Protocol --------------------------------------------------------------
LOCK_SEED = 2026              # fixes which 15% of SmartEars is the locked test set
EXPLORE_SEEDS = [7, 21, 42]   # each seed = one train/validation split + one training run per setup
GROUP_BY = "recording"        # clips of one source recording always stay together
TARGET_RECALL = 0.90

# ---- Input (same as Run 1) -------------------------------------------------
SAMPLE_RATE = 16000
CROP_SEC = 5.0
N_FFT, HOP_LENGTH, N_MELS = 400, 160, 64

# ---- Augmentation (same as Run 1) ------------------------------------------
AUG_SHIFT_PROB, AUG_SHIFT_SEC = 0.5, 0.5
AUG_TIME_MASK_PROB, AUG_FREQ_MASK_PROB = 0.4, 0.4
AUG_GAIN_PROB, AUG_GAIN_DB = 0.2, (-6.0, 6.0)

BATCH_SIZE = 16
LABEL_SMOOTHING = 0.02
DROPOUT = 0.3
USE_PRETRAINED_WEIGHTS = True # False = skip the ImageNet download (the "pretrained" setups then start from random weights)
QUICK_RUN = None              # e.g. 60 = use 60 clips per class, 2 epochs, 2 seeds (code check only)

# ---- The setups to compare -------------------------------------------------
# Each setup changes ONE thing relative to its parent, so the effect of that change can be read off.
RUN1 = {"norm": "none", "pretrained": False, "optimizer": "adam", "lr": 1e-4, "weight_decay": 1e-4,
        "schedule": "plateau", "max_epochs": 50, "patience": 8, "mixup": 0.0}
NORM = {"norm": "per_clip"}
PRETRAINED = {"pretrained": True}
RECIPE = {"optimizer": "adamw", "lr": 3e-4, "weight_decay": 1e-2, "schedule": "cosine",
          "max_epochs": 40, "patience": 10, "mixup": 0.2}


def setup(model, parent, change, *overrides):
    cfg = {"model": model, "parent": parent, "change": change, **RUN1}
    for o in overrides:
        cfg.update(o)
    return cfg


EXPERIMENTS = {
    "A0_cnn_bilstm_run1":        setup("cnn_bilstm", None, "Run 1 settings"),
    "A1_cnn_bilstm_norm":        setup("cnn_bilstm", "A0_cnn_bilstm_run1", "+ normalization", NORM),
    "A2_cnn_bilstm_norm_recipe": setup("cnn_bilstm", "A1_cnn_bilstm_norm", "+ recipe", NORM, RECIPE),
    "B0_mobilenet_run1":         setup("mobilenetv3_small", None, "Run 1 settings"),
    "B1_mobilenet_norm":         setup("mobilenetv3_small", "B0_mobilenet_run1", "+ normalization", NORM),
    "B2_mobilenet_norm_pre":     setup("mobilenetv3_small", "B1_mobilenet_norm", "+ ImageNet weights", NORM, PRETRAINED),
    "B3_mobilenet_norm_recipe":  setup("mobilenetv3_small", "B1_mobilenet_norm", "+ recipe", NORM, RECIPE),
    "B4_mobilenet_norm_pre_recipe": setup("mobilenetv3_small", "B2_mobilenet_norm_pre", "+ recipe", NORM, PRETRAINED, RECIPE),
    "C1_efficientnet_norm_pre_recipe": setup("efficientnet_b0", "B4_mobilenet_norm_pre_recipe", "larger pretrained model", NORM, PRETRAINED, RECIPE),
}

CLASS_NAMES = ["Non-distress", "Distress"]
CROP_SAMPLES = int(round(CROP_SEC * SAMPLE_RATE))
N_FRAMES = CROP_SAMPLES // HOP_LENGTH
NUM_WORKERS = 2 if os.name == "posix" else 0
if QUICK_RUN:
    EXPLORE_SEEDS = EXPLORE_SEEDS[:2]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
RUN_DIR = os.path.join(OUT_DIR, "runs")
os.makedirs(RUN_DIR, exist_ok=True)
print("Outputs:", OUT_DIR)
print(pd.DataFrame(EXPERIMENTS).T[["model", "parent", "change", "norm", "pretrained", "optimizer", "lr", "schedule", "mixup"]].to_string())

Mounted at /content/drive
Outputs: /content/drive/MyDrive/SONARA/run2_first_pass
                                             model                        parent                   change      norm pretrained optimizer      lr schedule mixup
A0_cnn_bilstm_run1                      cnn_bilstm                          None           Run 1 settings      none      False      adam  0.0001  plateau   0.0
A1_cnn_bilstm_norm                      cnn_bilstm            A0_cnn_bilstm_run1          + normalization  per_clip      False      adam  0.0001  plateau   0.0
A2_cnn_bilstm_norm_recipe               cnn_bilstm            A1_cnn_bilstm_norm                 + recipe  per_clip      False     adamw  0.0003   cosine   0.2
B0_mobilenet_run1                mobilenetv3_small                          None           Run 1 settings      none      False      adam  0.0001  plateau   0.0
B1_mobilenet_norm                mobilenetv3_small             B0_mobilenet_run1          + normalization  per_clip    

In [3]:
# Download the repository (code + audio). Takes several minutes the first time.
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
print("Repo folder:", REPO_DIR, "| exists:", os.path.isdir(REPO_DIR))

Cloning into '/content/SONARA-GITHUB'...
remote: Enumerating objects: 6377, done.
remote: Counting objects: 100% (6377/6377), done.
remote: Compressing objects: 100% (6377/6377), done.
remote: Total 6377 (delta 0), reused 6371 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6377/6377), 2.54 GiB | 29.45 MiB/s, done.
Updating files: 100% (6364/6364), done.
Repo folder: /content/SONARA-GITHUB | exists: True


## 2. Data

SmartEars only, loaded exactly as in Run 1: 16 kHz mono, the whole 5-second clip, binary labels (0 = Non-distress, 1 = Distress). PVSD is not loaded in this notebook.

In [4]:
SMARTEARS_NAME = re.compile(r"^(rpi\d+)-(\d{8})(\d{6})(?:-(\d+))?$")

rows = []
for folder, source in LABEL_MAP.items():
    for path in sorted(glob.glob(os.path.join(REPO_DIR, SMARTEARS_ROOT, folder, "*.wav"))):
        stem = os.path.splitext(os.path.basename(path))[0]
        m = SMARTEARS_NAME.match(stem)
        recording = f"{m.group(1)}-{m.group(2)}{m.group(3)}" if m else stem
        rows.append({"path": path, "file": f"{folder}/{os.path.basename(path)}", "original_label": folder,
                     "source": source, "label": int(source == "Distress"), "recording": recording})
manifest = pd.DataFrame(rows)
assert len(manifest) > 0, "No audio files found. Check REPO_DIR."
if QUICK_RUN:
    manifest = manifest.groupby("source").head(QUICK_RUN).reset_index(drop=True)
print(manifest.groupby(["original_label", "source", "label"]).size().to_string())


def load_audio(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != SAMPLE_RATE:
        g = gcd(int(sr), SAMPLE_RATE)
        wav = resample_poly(wav, SAMPLE_RATE // g, sr // g).astype(np.float32)
    return wav


t0 = time.time()
clips = []
for i, path in enumerate(manifest["path"]):
    clips.append(np.clip(load_audio(path) * 32767.0, -32768, 32767).astype(np.int16))
    if (i + 1) % 1000 == 0:
        print(f"  loaded {i + 1}/{len(manifest)} clips ({time.time() - t0:.0f}s)")
print(f"Loaded {len(clips)} clips in {time.time() - t0:.0f}s")

original_label  source    label
Healthy         Normal    0        2000
None            Noise     0        2000
Sick            Distress  1        2000
  loaded 1000/6000 clips (10s)
  loaded 2000/6000 clips (20s)
  loaded 3000/6000 clips (29s)
  loaded 4000/6000 clips (38s)
  loaded 5000/6000 clips (48s)
  loaded 6000/6000 clips (57s)
Loaded 6000 clips in 57s


## 3. Lock the test set, then split what is left

The locked test set is chosen once, from `LOCK_SEED`, and written to `locked_test_set.csv`. From this cell on, the code only ever indexes `dev_idx`, the other 85%.

For each exploration seed, the development data is split again into training (70% of all clips) and validation (15% of all clips). Both splits are stratified by source category and grouped by source recording.

In [5]:
def grouped_folds(indices, n_splits, seed):
    df = manifest.loc[indices]
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_of = np.zeros(len(df), dtype=int)
    for fold, (_, idx) in enumerate(sgkf.split(df, df["source"], df[GROUP_BY])):
        fold_of[idx] = fold
    return fold_of


all_idx = manifest.index.to_numpy()
fold_of = grouped_folds(all_idx, 20, LOCK_SEED)
locked_test_idx = all_idx[fold_of >= 17]          # 3 of 20 folds = 15%
dev_idx = all_idx[fold_of < 17]                   # the only clips this notebook uses
assert not set(manifest.loc[locked_test_idx, GROUP_BY]) & set(manifest.loc[dev_idx, GROUP_BY]), "group leakage"

lock_path = os.path.join(OUT_DIR, "locked_test_set.csv")
manifest.loc[locked_test_idx, ["file", "recording"]].to_csv(lock_path, index=False)
lock_id = hashlib.md5("\n".join(sorted(manifest.loc[locked_test_idx, "file"])).encode()).hexdigest()[:8]
print(f"Locked test set: {len(locked_test_idx)} clips, id {lock_id}, saved to {lock_path}")
print(f"Development set: {len(dev_idx)} clips")
DEV = set(dev_idx.tolist())
del locked_test_idx                                # nothing below can reach the locked clips by accident


def make_split(seed):
    fold_of = grouped_folds(dev_idx, 17, seed)
    split = {"train": dev_idx[fold_of < 14], "val": dev_idx[fold_of >= 14]}
    assert not set(manifest.loc[split["train"], GROUP_BY]) & set(manifest.loc[split["val"], GROUP_BY]), "group leakage"
    assert set(split["train"].tolist()) | set(split["val"].tolist()) <= DEV
    return split


for s in EXPLORE_SEEDS:
    sp = make_split(s)
    print(f"seed {s}: train {len(sp['train'])} {np.bincount(manifest.loc[sp['train'], 'label']).tolist()} | "
          f"val {len(sp['val'])} {np.bincount(manifest.loc[sp['val'], 'label']).tolist()}")

Locked test set: 871 clips, id bf636617, saved to /content/drive/MyDrive/SONARA/run2_first_pass/locked_test_set.csv
Development set: 5129 clips
seed 7: train 4218 [2808, 1410] | val 911 [595, 316]
seed 21: train 4235 [2812, 1423] | val 894 [591, 303]
seed 42: train 4219 [2818, 1401] | val 910 [585, 325]


## 4. Features, normalization and augmentation

The features are the same as in Run 1: mel power in dB plus delta and delta-delta, 3 × 64 × 500.

**Normalization (new).** With `norm = "per_clip"`, each channel of each clip has its own mean subtracted and is divided by its own standard deviation. A clip recorded louder or quieter then looks the same to the model. It uses only the clip itself, so no statistics are carried from one split to another.

Augmentation is unchanged from Run 1 and is applied before normalization. **Mixup (new, part of the recipe)** blends each training clip with another one from the same batch and blends their labels in the same proportion.

In [6]:
class LogMelDeltas(nn.Module):
    """Waveform (batch, samples) -> raw dB features (batch, 3, n_mels, frames)."""

    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                                        n_mels=N_MELS, center=True, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.deltas = torchaudio.transforms.ComputeDeltas()

    def features(self, wav):
        db = self.to_db(self.mel(wav))
        d1 = self.deltas(db)
        return torch.stack([db, d1, self.deltas(d1)], dim=1)

    def forward(self, wav, lengths):
        x = self.features(wav)[..., :N_FRAMES]
        for i in torch.nonzero(lengths < CROP_SAMPLES).flatten().tolist():   # clips a few ms short of 5 s
            f = self.features(wav[i:i + 1, :max(int(lengths[i]), N_FFT)])[0]
            t = min(f.shape[-1], N_FRAMES)
            x[i, 0] = f[0].min()
            x[i, 1:] = 0.0
            x[i, :, :, :t] = f[:, :, :t]
        return x


featurizer = LogMelDeltas().to(device).eval()


def normalize(x, mode):
    if mode == "per_clip":
        mean = x.mean(dim=(2, 3), keepdim=True)
        std = x.std(dim=(2, 3), keepdim=True)
        return (x - mean) / (std + 1e-5)
    return x


def augment_features(x):
    _, _, n_freq, n_time = x.shape
    max_shift = int(AUG_SHIFT_SEC * SAMPLE_RATE / HOP_LENGTH)
    for i in range(x.size(0)):
        if max_shift and random.random() < AUG_SHIFT_PROB:
            x[i] = torch.roll(x[i], random.randint(-max_shift, max_shift), dims=2)
        ch_mean = x[i].mean(dim=(1, 2), keepdim=True)
        if random.random() < AUG_TIME_MASK_PROB:
            width = random.randint(5, n_time // 8)
            t0 = random.randint(0, n_time - width)
            x[i, :, :, t0:t0 + width] = ch_mean
        if random.random() < AUG_FREQ_MASK_PROB:
            height = random.randint(3, n_freq // 8)
            f0 = random.randint(0, n_freq - height)
            x[i, :, f0:f0 + height, :] = ch_mean
        if random.random() < AUG_GAIN_PROB:
            x[i, 0] += random.uniform(*AUG_GAIN_DB)
    return x


class ClipDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)
        assert set(self.indices.tolist()) <= DEV, "a locked test clip was requested"
        self.labels = manifest["label"].to_numpy()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        seg = clips[idx][:CROP_SAMPLES].astype(np.float32) / 32768.0
        n = len(seg)
        if n < CROP_SAMPLES:
            seg = np.pad(seg, (0, CROP_SAMPLES - n))
        return torch.from_numpy(seg), int(self.labels[idx]), idx, n


def make_loader(indices, train, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ClipDataset(indices), batch_size=BATCH_SIZE, shuffle=train, num_workers=NUM_WORKERS,
                      generator=g, pin_memory=(device.type == "cuda"))

## 5. Models

- **CNN–BiLSTM**: the Run 1 model, unchanged (1,221,186 parameters).
- **MobileNetV3-Small**: about 1.5 million parameters. With `pretrained = True` it starts from ImageNet weights and only its last layer is new.
- **EfficientNet-B0**: about 4 million parameters, ImageNet weights. Larger, but still a size that runs on a Raspberry Pi 4.

ImageNet is a photo dataset. Its weights are used only as a starting point; every layer is then trained on SmartEars. The three feature channels (dB, delta, delta-delta) take the place of the three colour channels.

In [7]:
class CNN_BiLSTM(nn.Module):
    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, lstm_hidden=128, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.lstm = nn.LSTM(input_size=cnn_channels * (n_mels // 4), hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        b, c, fq, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * fq)
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(torch.cat([h_n[-2], h_n[-1]], dim=1)))


def build_model(name, pretrained):
    use_weights = "IMAGENET1K_V1" if (pretrained and USE_PRETRAINED_WEIGHTS) else None
    if name == "cnn_bilstm":
        return CNN_BiLSTM()
    if name == "mobilenetv3_small":
        m = torchvision.models.mobilenet_v3_small(weights=use_weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
        return m
    if name == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights=use_weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
        return m
    raise ValueError(name)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def time_ms(fn, runs=20):
    for _ in range(3):
        fn()
    ts = []
    for _ in range(runs):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    return round(float(np.median(ts)), 1)


# Size and a CPU timing proxy (this Colab CPU, NOT the Raspberry Pi 4) for the deployability check.
x_cpu = torch.randn(1, 3, N_MELS, N_FRAMES)
deploy = []
with torch.no_grad():
    for name in sorted({cfg["model"] for cfg in EXPERIMENTS.values()}):
        mdl = build_model(name, pretrained=False).eval()
        deploy.append({"model": name, "parameters": count_params(mdl), "size_MB": round(count_params(mdl) * 4 / 1e6, 2),
                       "cpu_ms_per_clip": time_ms(lambda: mdl(x_cpu))})
deploy = pd.DataFrame(deploy)
deploy.to_csv(os.path.join(OUT_DIR, "model_size_and_cpu_time.csv"), index=False)
print(f"CPU threads: {torch.get_num_threads()} (Colab proxy, not the Pi)")
print(deploy.to_string(index=False))

CPU threads: 1 (Colab proxy, not the Pi)
            model  parameters  size_MB  cpu_ms_per_clip
       cnn_bilstm     1221186     4.88             41.3
  efficientnet_b0     4010110    16.04             21.8
mobilenetv3_small     1519906     6.08              7.6


## 6. Training

The loss is the same as in Run 1: weighted cross-entropy with class weights from the training clips and label smoothing of 0.02. The best checkpoint is the epoch with the lowest validation loss, as in Run 1.

| | Run 1 settings | Recipe |
| :- | :- | :- |
| Optimizer | Adam, weight decay 1e-4 | AdamW, weight decay 1e-2 |
| Learning rate | 1e-4, halved when validation loss stalls | 3e-4, 2 warm-up epochs, then a cosine decay |
| Epochs | at most 50, patience 8 | at most 40, patience 10 |
| Mixup | none | Beta(0.2, 0.2) |

The recipe is one alternative fixed in advance, not the result of a search.

In [8]:
def run_epoch(model, loader, cfg, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total, probs, labels = 0.0, 0, [], []
    with torch.set_grad_enabled(training):
        for wav, y, _, n in loader:
            wav, y = wav.to(device), y.to(device)
            with torch.no_grad():
                x = featurizer(wav, n)
                if training:
                    x = augment_features(x)
                x = normalize(x, cfg["norm"])
            if training and cfg["mixup"] > 0:
                lam = float(np.random.beta(cfg["mixup"], cfg["mixup"]))
                perm = torch.randperm(x.size(0), device=device)
                logits = model(lam * x + (1 - lam) * x[perm])
                loss = lam * criterion(logits, y) + (1 - lam) * criterion(logits, y[perm])
            else:
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            else:
                probs.append(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
                labels.append(y.cpu().numpy())
            total_loss += loss.item() * y.size(0)
            total += y.size(0)
    if training:
        return total_loss / total, None, None
    return total_loss / total, np.concatenate(probs), np.concatenate(labels)


def train_setup(name, seed, split):
    cfg = EXPERIMENTS[name]
    max_epochs = 2 if QUICK_RUN else cfg["max_epochs"]
    set_seed(seed)
    model = build_model(cfg["model"], cfg["pretrained"]).to(device)
    counts = np.bincount(manifest.loc[split["train"], "label"], minlength=2)
    weights = torch.tensor(counts.mean() / counts, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
    if cfg["optimizer"] == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if cfg["schedule"] == "cosine":
        warm = 2
        scheduler = LambdaLR(optimizer, lambda e: (e + 1) / warm if e < warm else
                             max(0.02, 0.5 * (1 + math.cos(math.pi * (e - warm) / max(1, max_epochs - warm)))))
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)
    train_loader = make_loader(split["train"], True, seed)
    val_loader = make_loader(split["val"], False, seed)

    history, best = [], {"loss": float("inf")}
    stale = 0
    for epoch in range(1, max_epochs + 1):
        train_loss, _, _ = run_epoch(model, train_loader, cfg, criterion, optimizer)
        val_loss, p_val, y_val = run_epoch(model, val_loader, cfg, criterion)
        scheduler.step(val_loss) if cfg["schedule"] == "plateau" else scheduler.step()
        improved = val_loss < best["loss"] - 1e-4
        if improved:
            best, stale = {"loss": val_loss, "epoch": epoch, "p_val": p_val, "y_val": y_val}, 0
        else:
            stale += 1
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_auc": roc_auc_score(y_val, p_val), "best": improved})
        print(f"  [{name} seed {seed}] epoch {epoch:02d} train {train_loss:.4f} val {val_loss:.4f} "
              f"auc {history[-1]['val_auc']:.4f}{' *' if improved else ''}")
        if stale >= cfg["patience"]:
            break
    return best, pd.DataFrame(history), count_params(model)


def choose_threshold(y, p, target=TARGET_RECALL):
    ok = [t for t in np.round(np.arange(0.01, 0.995, 0.005), 3)
          if recall_score(y, (p >= t).astype(int), zero_division=0) >= target]
    return float(max(ok)) if ok else 0.5


def validation_metrics(y, p, sources):
    pred = (p >= 0.5).astype(int)
    thr = choose_threshold(y, p)
    flagged = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, flagged, labels=[0, 1]).ravel()
    return {"val_roc_auc": roc_auc_score(y, p), "val_pr_auc": average_precision_score(y, p),
            "val_accuracy_at_0.5": accuracy_score(y, pred),
            "val_distress_f1_at_0.5": f1_score(y, pred, zero_division=0),
            "val_distress_recall_at_0.5": recall_score(y, pred, zero_division=0),
            "threshold_for_recall_0.90": thr,
            "val_false_alarm_rate_at_recall_0.90": fp / max(fp + tn, 1),
            "val_false_alarms_normal": float(flagged[sources == "Normal"].mean()),
            "val_false_alarms_noise": float(flagged[sources == "Noise"].mean())}

## 7. Run every setup on every seed

Each finished run writes a small JSON file, its validation predictions and its training history to `OUT_DIR/runs`. Finished runs are skipped when this cell is run again, so after a disconnect just run the notebook again.

In [9]:
for seed in EXPLORE_SEEDS:
    split = make_split(seed)
    for name, cfg in EXPERIMENTS.items():
        base = os.path.join(RUN_DIR, f"{name}_seed{seed}")
        if os.path.exists(base + ".json"):
            print(f"skip (done): {name}_seed{seed}")
            continue
        t0 = time.time()
        best, history, n_params = train_setup(name, seed, split)
        sources = manifest.loc[split["val"], "source"].to_numpy()
        result = {"setup": name, "seed": seed, "lock_id": lock_id,
                  **{k: v for k, v in cfg.items()}, "parameters": n_params,
                  "epochs_run": len(history), "best_epoch": best["epoch"], "best_val_loss": best["loss"],
                  "train_minutes": (time.time() - t0) / 60,
                  **validation_metrics(best["y_val"], best["p_val"], sources)}
        np.savez(base + "_val_predictions.npz", p_val=best["p_val"], y_val=best["y_val"], rows_val=split["val"])
        history.to_csv(base + "_history.csv", index=False)
        with open(base + ".json", "w") as f:        # written last: marks the run as complete
            json.dump(result, f, indent=1, default=float)
        print(f"DONE {name}_seed{seed}: val AUC {result['val_roc_auc']:.4f} | acc@0.5 {result['val_accuracy_at_0.5']:.4f} | "
              f"false alarms at 0.90 recall {result['val_false_alarm_rate_at_recall_0.90']:.4f} | {result['train_minutes']:.1f} min\n")

  [A0_cnn_bilstm_run1 seed 7] epoch 01 train 0.6200 val 0.5469 auc 0.8528 *
  [A0_cnn_bilstm_run1 seed 7] epoch 02 train 0.5212 val 0.4464 auc 0.8849 *
  [A0_cnn_bilstm_run1 seed 7] epoch 03 train 0.5029 val 0.5069 auc 0.8861
  [A0_cnn_bilstm_run1 seed 7] epoch 04 train 0.4854 val 0.4823 auc 0.8834
  [A0_cnn_bilstm_run1 seed 7] epoch 05 train 0.4681 val 0.4449 auc 0.8908 *
  [A0_cnn_bilstm_run1 seed 7] epoch 06 train 0.4773 val 0.4666 auc 0.8831
  [A0_cnn_bilstm_run1 seed 7] epoch 07 train 0.4627 val 0.4352 auc 0.8953 *
  [A0_cnn_bilstm_run1 seed 7] epoch 08 train 0.4566 val 0.4657 auc 0.8884
  [A0_cnn_bilstm_run1 seed 7] epoch 09 train 0.4390 val 0.4116 auc 0.9037 *
  [A0_cnn_bilstm_run1 seed 7] epoch 10 train 0.4374 val 0.4631 auc 0.9043
  [A0_cnn_bilstm_run1 seed 7] epoch 11 train 0.4286 val 0.4024 auc 0.9045 *
  [A0_cnn_bilstm_run1 seed 7] epoch 12 train 0.4250 val 0.3840 auc 0.9139 *
  [A0_cnn_bilstm_run1 seed 7] epoch 13 train 0.4137 val 0.3839 auc 0.9129 *
  [A0_cnn_bilstm_run1 

100%|██████████| 9.83M/9.83M [00:00<00:00, 147MB/s]


  [B2_mobilenet_norm_pre seed 7] epoch 01 train 0.5253 val 0.4658 auc 0.8772 *
  [B2_mobilenet_norm_pre seed 7] epoch 02 train 0.4365 val 0.4280 auc 0.8964 *
  [B2_mobilenet_norm_pre seed 7] epoch 03 train 0.3952 val 0.4854 auc 0.9058
  [B2_mobilenet_norm_pre seed 7] epoch 04 train 0.3712 val 0.3787 auc 0.9194 *
  [B2_mobilenet_norm_pre seed 7] epoch 05 train 0.3414 val 0.4192 auc 0.9238
  [B2_mobilenet_norm_pre seed 7] epoch 06 train 0.3397 val 0.3720 auc 0.9269 *
  [B2_mobilenet_norm_pre seed 7] epoch 07 train 0.3088 val 0.3695 auc 0.9384 *
  [B2_mobilenet_norm_pre seed 7] epoch 08 train 0.3035 val 0.3503 auc 0.9364 *
  [B2_mobilenet_norm_pre seed 7] epoch 09 train 0.2782 val 0.3508 auc 0.9415
  [B2_mobilenet_norm_pre seed 7] epoch 10 train 0.2691 val 0.3548 auc 0.9386
  [B2_mobilenet_norm_pre seed 7] epoch 11 train 0.2640 val 0.3433 auc 0.9438 *
  [B2_mobilenet_norm_pre seed 7] epoch 12 train 0.2553 val 0.3364 auc 0.9448 *
  [B2_mobilenet_norm_pre seed 7] epoch 13 train 0.2357 val 0

100%|██████████| 20.5M/20.5M [00:00<00:00, 120MB/s]


  [C1_efficientnet_norm_pre_recipe seed 7] epoch 01 train 0.5521 val 0.4312 auc 0.9002 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 02 train 0.4873 val 0.3773 auc 0.9271 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 03 train 0.4531 val 0.3694 auc 0.9336 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 04 train 0.4198 val 0.3477 auc 0.9440 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 05 train 0.3920 val 0.3186 auc 0.9494 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 06 train 0.3863 val 0.3507 auc 0.9365
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 07 train 0.3583 val 0.3174 auc 0.9502 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 08 train 0.3391 val 0.3100 auc 0.9483 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 09 train 0.3321 val 0.3248 auc 0.9412
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 10 train 0.3270 val 0.3050 auc 0.9519 *
  [C1_efficientnet_norm_pre_recipe seed 7] epoch 11 train 0.3088 val 0.3316 auc 0.9421
  [C1_efficientnet_norm_pre

## 8. Validation results

All numbers in this part are **validation** results, mean ± standard deviation over the exploration seeds.

- **ROC-AUC** is the selection metric. It does not depend on a threshold.
- **False alarms at 0.90 recall** is the share of Non-distress clips flagged when the threshold is set to catch 90% of Distress clips. This is the operating point SONARA cares about; lower is better. Run 1's CNN–BiLSTM flagged about 24% on its test sets.

In [10]:
def pm(s, digits=3):
    return f"{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}" if len(s) > 1 else f"{s.mean():.{digits}f}"


runs = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(os.path.join(RUN_DIR, "*.json")))])
runs = runs[runs["setup"].isin(EXPERIMENTS) & runs["seed"].isin(EXPLORE_SEEDS) & (runs["lock_id"] == lock_id)]
runs.to_csv(os.path.join(OUT_DIR, "all_runs.csv"), index=False)

metric_names = {"val_roc_auc": "ROC-AUC", "val_pr_auc": "PR-AUC", "val_accuracy_at_0.5": "Accuracy @0.5",
                "val_distress_f1_at_0.5": "Distress F1 @0.5",
                "val_false_alarm_rate_at_recall_0.90": "False alarms @0.90 recall",
                "val_false_alarms_normal": "…on Normal", "val_false_alarms_noise": "…on Noise"}
summary = []
for name, cfg in EXPERIMENTS.items():
    g = runs[runs["setup"] == name]
    if len(g) == 0:
        continue
    summary.append({"Setup": name, "Change": cfg["change"], "Seeds": len(g),
                    "Parameters": f"{int(g['parameters'].iloc[0]):,}", "Best epoch": pm(g["best_epoch"], 1),
                    **{label: pm(g[col]) for col, label in metric_names.items()}})
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_DIR, "first_pass_validation_summary.csv"), index=False)
pd.set_option("display.width", 250)
print("VALIDATION results (not test results)")
print(summary.to_string(index=False))

VALIDATION results (not test results)
                          Setup                  Change  Seeds Parameters  Best epoch       ROC-AUC        PR-AUC Accuracy @0.5 Distress F1 @0.5 False alarms @0.90 recall    …on Normal     …on Noise
             A0_cnn_bilstm_run1          Run 1 settings      3  1,221,186  35.7 ± 8.1 0.944 ± 0.005 0.897 ± 0.015 0.882 ± 0.005    0.837 ± 0.007             0.141 ± 0.013 0.201 ± 0.017 0.073 ± 0.017
             A1_cnn_bilstm_norm         + normalization      3  1,221,186 32.3 ± 13.7 0.934 ± 0.008 0.880 ± 0.020 0.877 ± 0.014    0.830 ± 0.016             0.164 ± 0.029 0.233 ± 0.025 0.086 ± 0.033
      A2_cnn_bilstm_norm_recipe                + recipe      3  1,221,186  36.0 ± 4.4 0.946 ± 0.003 0.904 ± 0.004 0.886 ± 0.007    0.842 ± 0.004             0.133 ± 0.019 0.195 ± 0.030 0.065 ± 0.015
              B0_mobilenet_run1          Run 1 settings      3  1,519,906  23.7 ± 2.3 0.936 ± 0.001 0.888 ± 0.009 0.874 ± 0.010    0.823 ± 0.015             0.178 ± 0

In [11]:
# Effect of each change: a setup compared with its parent, seed by seed (same seed = same split).
comparisons = []
for name, cfg in EXPERIMENTS.items():
    if not cfg["parent"]:
        continue
    a = runs[runs["setup"] == name].set_index("seed")
    b = runs[runs["setup"] == cfg["parent"]].set_index("seed")
    common = a.index.intersection(b.index)
    if len(common) == 0:
        continue
    d_auc = a.loc[common, "val_roc_auc"] - b.loc[common, "val_roc_auc"]
    d_fa = a.loc[common, "val_false_alarm_rate_at_recall_0.90"] - b.loc[common, "val_false_alarm_rate_at_recall_0.90"]
    comparisons.append({"Setup": name, "Compared with": cfg["parent"], "Change": cfg["change"], "Seeds": len(common),
                        "ROC-AUC change": round(d_auc.mean(), 4), "Seeds better (AUC)": int((d_auc > 0).sum()),
                        "False-alarm change": round(d_fa.mean(), 4), "Seeds better (false alarms)": int((d_fa < 0).sum())})
comparisons = pd.DataFrame(comparisons)
comparisons.to_csv(os.path.join(OUT_DIR, "first_pass_comparisons.csv"), index=False)
print(comparisons.to_string(index=False))
print("\nA positive ROC-AUC change and a negative false-alarm change are improvements. Count a change as "
      f"helping only if it is better on most of the {len(EXPLORE_SEEDS)} seeds; otherwise report no clear effect.")

                          Setup                Compared with                  Change  Seeds  ROC-AUC change  Seeds better (AUC)  False-alarm change  Seeds better (false alarms)
             A1_cnn_bilstm_norm           A0_cnn_bilstm_run1         + normalization      3         -0.0099                   0              0.0232                            0
      A2_cnn_bilstm_norm_recipe           A1_cnn_bilstm_norm                + recipe      3          0.0122                   3             -0.0307                            2
              B1_mobilenet_norm            B0_mobilenet_run1         + normalization      3         -0.0353                   0              0.1063                            0
          B2_mobilenet_norm_pre            B1_mobilenet_norm      + ImageNet weights      3          0.0545                   3             -0.1701                            3
       B3_mobilenet_norm_recipe            B1_mobilenet_norm                + recipe      3          0.0373        

In [12]:
# Ranking by the selection rule fixed in advance: mean validation ROC-AUC.
ranking = (runs.groupby("setup")
           .agg(model=("model", "first"), seeds=("seed", "size"), val_roc_auc=("val_roc_auc", "mean"),
                false_alarms_at_090=("val_false_alarm_rate_at_recall_0.90", "mean"),
                minutes_per_run=("train_minutes", "mean"))
           .reset_index()
           .merge(deploy[["model", "parameters", "cpu_ms_per_clip"]], on="model")
           .sort_values("val_roc_auc", ascending=False).round(4))
ranking.to_csv(os.path.join(OUT_DIR, "first_pass_ranking.csv"), index=False)
print(ranking.to_string(index=False))
incomplete = ranking[ranking["seeds"] < len(EXPLORE_SEEDS)]["setup"].tolist()
if incomplete:
    print("\nNot all seeds finished for:", incomplete, "- rerun the notebook before reading the ranking.")
print(f"\nLeading setup on validation ROC-AUC: {ranking.iloc[0]['setup']}")
print("This is a validation result. The locked test set (id " + lock_id + ") and PVSD have not been used.")

                          setup             model  seeds  val_roc_auc  false_alarms_at_090  minutes_per_run  parameters  cpu_ms_per_clip
C1_efficientnet_norm_pre_recipe   efficientnet_b0      3       0.9636               0.0740          14.4343     4010110             21.8
   B4_mobilenet_norm_pre_recipe mobilenetv3_small      3       0.9615               0.0863           6.5226     1519906              7.6
          B2_mobilenet_norm_pre mobilenetv3_small      3       0.9551               0.1141           4.7199     1519906              7.6
      A2_cnn_bilstm_norm_recipe        cnn_bilstm      3       0.9462               0.1332           8.4280     1221186             41.3
             A0_cnn_bilstm_run1        cnn_bilstm      3       0.9440               0.1407           8.9063     1221186             41.3
       B3_mobilenet_norm_recipe mobilenetv3_small      3       0.9379               0.1548           7.2100     1519906              7.6
              B0_mobilenet_run1 mobilenet

## 9. What to do with these results

1. Copy `first_pass_validation_summary.csv`, `first_pass_comparisons.csv` and `first_pass_ranking.csv` from `OUT_DIR` into the repo under `results/`, next to the Run 1 folder, together with this executed notebook.
2. Record which changes helped on most seeds and which did not. Changes that did not help are part of the record too.
3. Keep `locked_test_set.csv`. The second pass and the final evaluation must use the same locked clips.
4. Do not report these validation numbers as the system's performance. The paper's results come from the final evaluation on the locked test set and PVSD.